In [1]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm.auto import tqdm

# -------------------------
# Config
# -------------------------
CSV_PATH = "voxpopuli_with_mfcc.csv"
CHECKPOINT_DIR = "mlp_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-3
NUM_WORKERS = 0  # safer on Windows

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

# -------------------------
# Load CSV + labels
# -------------------------
df = pd.read_csv(CSV_PATH)
mfcc_cols = [col for col in df.columns if col.startswith("mfcc_")]

label_col = "lang"
le = LabelEncoder()
df['label_id'] = le.fit_transform(df[label_col])
num_labels = len(le.classes_)

train_df, val_df = train_test_split(
    df, test_size=0.12, random_state=42, stratify=df['label_id']
)

# -------------------------
# Dataset
# -------------------------
class MFCCDataset(Dataset):
    def __init__(self, df, mfcc_cols):
        self.features = df[mfcc_cols].values.astype("float32")
        self.labels = df['label_id'].values.astype("int64")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.features[idx], dtype=torch.float32)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

# DataLoaders
train_loader = DataLoader(MFCCDataset(train_df, mfcc_cols),
                          batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)

val_loader = DataLoader(MFCCDataset(val_df, mfcc_cols),
                        batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)

# -------------------------
# MLP Model
# -------------------------
class MFCC_MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.model(x)

model = MFCC_MLP(input_dim=len(mfcc_cols), hidden_dim=128, num_classes=num_labels).to(DEVICE)

# -------------------------
# Optimizer + loss
# -------------------------
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

# -------------------------
# Training loop
# -------------------------
best_val_acc = 0.0

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for X, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        X = X.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    # Validation
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in val_loader:
            X = X.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            logits = model(X)
            preds = torch.argmax(logits, dim=-1)
            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(y.cpu().tolist())

    val_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch} | Train Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_model.pt"))
        print("✅ Saved new best model.")

# -------------------------
# Final Evaluation
# -------------------------
model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, "best_model.pt")))
model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for X, y in val_loader:
        X = X.to(DEVICE)
        y = y.to(DEVICE)
        logits = model(X)
        preds = torch.argmax(logits, dim=-1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(y.cpu().tolist())

print("\n✅ Final Evaluation:")
print(classification_report(all_labels, all_preds, target_names=list(le.classes_)))
print("Confusion Matrix:\n", confusion_matrix(all_labels, all_preds))


c:\Users\Pc\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


Epoch 1/20: 100%|██████████| 207/207 [00:00<00:00, 250.95it/s]


Epoch 1 | Train Loss: 1.2227 | Val Acc: 0.3228
✅ Saved new best model.


Epoch 2/20: 100%|██████████| 207/207 [00:00<00:00, 316.30it/s]


...

Epoch 19/20: 100%|██████████| 207/207 [00:00<00:00, 311.88it/s]


Epoch 19 | Train Loss: 1.0982 | Val Acc: 0.3322


Epoch 20/20: 100%|██████████| 207/207 [00:00<00:00, 313.45it/s]


Epoch 20 | Train Loss: 1.0979 | Val Acc: 0.3144

✅ Final Evaluation:


TypeError: object of type 'numpy.int64' has no len()